# 06. 결과 분석

세 방법의 비교표, 국가별 정확도, 혼동행렬(제로샷 vs 부분 파인튜닝), 사례 분석. 런타임: CPU로 충분하다.

03\~05 노트북을 실행해 `results/`에 세 방법의 결과가 있어야 한다.

In [ ]:
# 공통 셋업: 저장소 clone, 패키지 설치, Drive 연결
import os, sys
REPO_URL = "https://github.com/Cranezero/aix-assignment-pr.git"
REPO_DIR = "/content/aix-assignment-pr"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from src import config
from src.data import prepare_images
from src.metrics import load_pred, plot_confusion, top_confusions
from src.utils import export_outputs, restore_outputs

restore_outputs()  # results/, assets/ 폴더 생성 + Drive에 저장된 이전 결과 가져오기

image_dir = prepare_images()
METHODS = {"zeroshot": "Zero-shot", "linear_probe": "Linear probe", "finetune": "Partial fine-tuning"}
metrics = {m: json.load(open(config.RESULTS_DIR / f"{m}_metrics.json")) for m in METHODS}
preds = {m: load_pred(m) for m in METHODS}
assert all((preds[m]["id"].values == preds["zeroshot"]["id"].values).all() for m in METHODS)

## 비교표

In [ ]:
comparison = pd.DataFrame([{
    "method": name, "top1": metrics[m]["top1"], "top5": metrics[m]["top5"], "macro": metrics[m]["macro"],
    "train_time_min": metrics[m].get("train_time_sec", 0) / 60,
} for m, name in METHODS.items()])
comparison.to_csv(config.RESULTS_DIR / "comparison.csv", index=False)
display(comparison.round(4))

## 국가별 정확도

In [ ]:
per_country = pd.DataFrame({name: metrics[m]["per_country"] for m, name in METHODS.items()}).loc[config.CODES]
per_country.to_csv(config.RESULTS_DIR / "per_country.csv")
order = per_country["Zero-shot"].sort_values().index
ax = per_country.loc[order].plot.barh(figsize=(8, 10), width=0.8)
ax.set_xlim(0, 1); ax.set_xlabel("accuracy"); ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(config.ASSETS_DIR / "per_country.png", dpi=120, bbox_inches="tight")
plt.show()
display(per_country.loc[order[::-1]].round(3))

## 혼동행렬: 제로샷 vs 부분 파인튜닝

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8.5))
for ax, m in zip(axes, ["zeroshot", "finetune"]):
    plot_confusion(preds[m]["country"], preds[m]["pred"], ax=ax, title=f"{METHODS[m]} (row-normalized)")
plt.tight_layout()
plt.savefig(config.ASSETS_DIR / "confusion_matrix.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# 제로샷에서 많이 혼동한 쌍이 학습 후 어떻게 바뀌었는가 (rate = 정답 국가 테스트 장수 대비 비율)
zs_top = top_confusions(preds["zeroshot"]["country"], preds["zeroshot"]["pred"], n=10)
for m in ["linear_probe", "finetune"]:
    d = preds[m]
    rate = {(t, p): ((d["country"] == t) & (d["pred"] == p)).sum() / (d["country"] == t).sum()
            for t, p in zip(zs_top["true"], zs_top["pred"])}
    zs_top[f"rate_{m}"] = [rate[(t, p)] for t, p in zip(zs_top["true"], zs_top["pred"])]
zs_top.to_csv(config.RESULTS_DIR / "confusion_pairs.csv", index=False)
display(zs_top.round(3))

In [ ]:
ft_top = top_confusions(preds["finetune"]["country"], preds["finetune"]["pred"], n=10)
display(ft_top.round(3))

## 사례 분석

In [ ]:
zs, lp, ft = preds["zeroshot"], preds["linear_probe"], preds["finetune"]
fixed = zs[(zs["pred"] != zs["country"]) & (ft["pred"] == ft["country"])].index
all_wrong = zs[(zs["pred"] != zs["country"]) & (lp["pred"] != lp["country"]) & (ft["pred"] != ft["country"])].index
print("제로샷 오답 -> 파인튜닝 정답:", len(fixed), "/ 모든 방법 오답:", len(all_wrong))

def case_grid(idx, name, n=8, seed=config.SEED):
    """사례 그림은 assets/{name}.png, 고른 사진의 id와 방법별 예측은 results/{name}.csv로 저장한다."""
    rng = np.random.default_rng(seed)
    pick = rng.choice(idx, size=min(n, len(idx)), replace=False)
    fig, axes = plt.subplots(2, 4, figsize=(18, 7.5))
    for ax, i in zip(axes.flat, pick):
        ax.imshow(Image.open(image_dir / f"{zs.loc[i, 'id']}.jpg")); ax.axis("off")
        ax.set_title(f"True {zs.loc[i, 'country']} | ZS {zs.loc[i, 'pred']} | "
                     f"LP {lp.loc[i, 'pred']} | FT {ft.loc[i, 'pred']}", fontsize=10)
    for ax in list(axes.flat)[len(pick):]:
        ax.axis("off")
    plt.tight_layout(); plt.savefig(config.ASSETS_DIR / f"{name}.png", dpi=100, bbox_inches="tight"); plt.show()
    table = zs.loc[pick, ["id", "country"]].assign(zs=zs.loc[pick, "pred"], lp=lp.loc[pick, "pred"],
                                                    ft=ft.loc[pick, "pred"])
    table.to_csv(config.RESULTS_DIR / f"{name}.csv", index=False)
    return table

display(case_grid(fixed, "case_fixed"))

In [ ]:
display(case_grid(all_wrong, "case_all_wrong"))

In [ ]:
# 모든 방법이 틀린 사진의 국가 분포
display(zs.loc[all_wrong, "country"].value_counts().to_frame("count").T)

In [ ]:
export_outputs()